# Expressions régulières avec le module `re`

Ce notebook pratique les expressions régulières sur de petits textes, sur un journal de connexions, puis sur un corpus de près de 4 000 e-mails frauduleux :

1. Valider des saisies
2. Des numéros de téléphone au même format
3. Des liens HTML en Markdown
4. Nettoyer un texte copié d'un PDF
5. Les chaînes entre guillemets
6. Des tentatives d'intrusion
7. Un corpus d'e-mails frauduleux
8. Nettoyer un texte accentué (facultatif)

Les exercices sont indépendants, sauf les étapes du 7, qui s'enchaînent. Chaque exercice a une cellule pour votre code, puis des conseils supplémentaires et sa solution, repliés : ouvrez-les si besoin.

Pour mettre au point un motif, https://regex101.com/ (saveur « Python ») explique chacune de ses parties et montre ce qu'il reconnaît. Si les bases vous manquent, le tutoriel https://regexone.com/ les reprend pas à pas.

## 1. Valider des saisies

Une plaque d'immatriculation française s'écrit `AB-123-CD` : deux lettres majuscules, un tiret, trois chiffres, un tiret, deux lettres majuscules, soit le motif `motif` ci-dessous.

Pour chaque chaîne de `plaques`, affichez si `re.search` et `re.fullmatch` y trouvent le motif (leur résultat est `None` ou un objet `Match`). Vous devez obtenir :

```
AB-123-CD    search: True  fullmatch: True
ab-123-cd    search: False fullmatch: False
xAB-123-CDx  search: True  fullmatch: False
AB-1234-CD   search: False fullmatch: False
AB-123-CDE   search: True  fullmatch: False
GH-007-XY    search: True  fullmatch: True
```

Laquelle des deux fonctions sert à valider une saisie ? Puis, avec `re.compile` et le flag `re.IGNORECASE`, construisez `valide`, un motif compilé qui accepte aussi les minuscules, et affichez la liste des plaques que `valide.fullmatch` accepte : `['AB-123-CD', 'ab-123-cd', 'GH-007-XY']`.

In [ ]:
import re

plaques = [
  "AB-123-CD",
  "ab-123-cd",
  "xAB-123-CDx",
  "AB-1234-CD",
  "AB-123-CDE",
  "GH-007-XY",
]
motif = r"[A-Z]{2}-\d{3}-[A-Z]{2}"

# Votre code ici

### Conseils supplémentaires

- `re.search(motif, plaque) is not None` vaut `True` si le motif est trouvé n'importe où dans `plaque` ; `re.fullmatch`, seulement s'il couvre toute la chaîne
- dans une f-string, `{plaque:12}` complète `plaque` par des espaces jusqu'à 12 caractères, et `{s!s:5}` fait de même pour le booléen `s`
- `re.compile(motif, re.IGNORECASE)` renvoie un motif compilé, dont la méthode `fullmatch(plaque)` s'utilise comme `re.fullmatch`

Un squelette à copier dans la cellule de l'exercice, en remplaçant chaque `...` :

```python
for plaque in plaques:
  s = ... is not None
  f = ... is not None
  print(f"{plaque:12} search: {s!s:5} fullmatch: {f}")

valide = re.compile(motif, ...)
print([plaque for plaque in plaques if ...])
```

### Solution

In [ ]:
for plaque in plaques:
  s = re.search(motif, plaque) is not None
  f = re.fullmatch(motif, plaque) is not None
  print(f"{plaque:12} search: {s!s:5} fullmatch: {f}")

# re.fullmatch valide une saisie : re.search accepte aussi une plaque entourée
# d'autres caractères (xAB-123-CDx, AB-123-CDE).

valide = re.compile(motif, re.IGNORECASE)
print([plaque for plaque in plaques if valide.fullmatch(plaque)])

## 2. Des numéros de téléphone au même format

Un formulaire a reçu les numéros de `numeros`, écrits de six façons. Écrivez une fonction `normalise(numero)` qui renvoie le numéro au format `06 39 98 12 34` : dix chiffres groupés par deux, l'indicatif `+33` remplacé par un `0`. Affichez ensuite le numéro normalisé de chaque élément de `numeros`. Vous devez obtenir :

```
06 39 98 12 34
06 39 98 12 34
06 39 98 12 34
06 39 98 12 34
01 99 00 56 78
01 99 00 56 78
```

In [ ]:
numeros = [
  "06 39 98 12 34",
  "06.39.98.12.34",
  "0639981234",
  "+33 6 39 98 12 34",
  "01-99-00-56-78",
  "+33 (0)1 99 00 56 78",
]

# Votre code ici

### Conseils supplémentaires

Trois remplacements dans `normalise` :

- `\D` reconnaît ce qui n'est pas un chiffre : le supprimer ne garde que les chiffres, et `"+33 (0)1 99 00 56 78"` devient `"330199005678"`
- un `33` en début de chaîne, suivi ou non d'un `0`, devient un `0` : `^` ancre le motif au début, `?` rend le `0` facultatif
- cinq groupes de deux chiffres, `(\d{2})` cinq fois, se réécrivent `\1 \2 \3 \4 \5`

Exemple : `normalise("+33 (0)1 99 00 56 78")` renvoie `"01 99 00 56 78"`.

Un squelette à copier dans la cellule de l'exercice, en remplaçant chaque `...` :

```python
def normalise(numero: str) -> str:
  chiffres = re.sub(r"\D", "", numero)
  chiffres = re.sub(r"...", "0", chiffres)
  return re.sub(r"...", r"\1 \2 \3 \4 \5", chiffres)


for numero in numeros:
  print(normalise(numero))
```

### Solution

In [ ]:
def normalise(numero: str) -> str:
  chiffres = re.sub(r"\D", "", numero)
  chiffres = re.sub(r"^330?", "0", chiffres)
  return re.sub(r"(\d{2})(\d{2})(\d{2})(\d{2})(\d{2})", r"\1 \2 \3 \4 \5", chiffres)


for numero in numeros:
  print(normalise(numero))

## 3. Des liens HTML en Markdown

En HTML, un lien s'écrit `<a href="adresse">texte</a>` ; en Markdown, `[texte](adresse)`. Avec `re.sub`, réécrivez en Markdown les deux liens de `html`, et affichez le résultat :

```
<p>Voir [la doc de re](https://docs.python.org/fr/3/library/re.html) et [regex101](https://regex101.com/).</p>
```

Essayez d'abord le motif `<a href="(.*)">(.*)</a>` : pourquoi ne donne-t-il pas ce résultat ? Corrigez-le.

In [ ]:
html = (
  '<p>Voir <a href="https://docs.python.org/fr/3/library/re.html">la doc de re</a>'
  ' et <a href="https://regex101.com/">regex101</a>.</p>'
)

# Votre code ici

### Conseils supplémentaires

Le remplacement réutilise les groupes dans l'autre ordre : `r"[\2](\1)"`.

`*` est gourmand : il répète le plus possible, et le premier `(.*)` va jusqu'au dernier `">` de la ligne, à travers le premier lien. Deux corrections :

- interdire le guillemet dans l'adresse et le chevron dans le texte, avec des classes de caractères niées : `[^"]*` et `[^<]*`
- ou rendre la répétition non gourmande, `.*?`, qui répète le moins possible

Un squelette à copier dans la cellule de l'exercice, en remplaçant chaque `...` :

```python
print(re.sub(r'<a href="(...)">(...)</a>', r"[\2](\1)", html))
```

### Solution

In [ ]:
# Avec des classes de caractères niées
print(re.sub(r'<a href="([^"]*)">([^<]*)</a>', r"[\2](\1)", html))
# Avec des répétitions non gourmandes
print(re.sub(r'<a href="(.*?)">(.*?)</a>', r"[\2](\1)", html))

## 4. Nettoyer un texte copié d'un PDF

Le texte `texte`, copié d'un PDF, a des blancs en trop : plusieurs espaces ou une tabulation entre deux mots, une espace avant les virgules et les points, des espaces en fin de ligne. Avec `re.sub`, construisez `propre`, le texte sans ces blancs, et affichez-le. Vous devez obtenir :

```
Le module re cherche des motifs.
Il remplace, découpe et extrait
les morceaux d'un texte.
```

Gardez les retours à la ligne : seuls les espaces et les tabulations sont en trop.

In [ ]:
import re

texte = (
  "Le  module re   cherche des motifs .\n"
  "Il\tremplace , découpe et extrait   \n"
  "les morceaux d'un texte ."
)

# Votre code ici

### Conseils supplémentaires

Trois remplacements, l'un après l'autre :

- `[ \t]+` reconnaît une suite d'espaces et de tabulations, à remplacer par une seule espace. `\s+` irait trop loin : il reconnaît aussi le saut de ligne `\n`, et recollerait les lignes
- une espace suivie d'une virgule ou d'un point : capturez la ponctuation dans un groupe, et remettez-la seule avec `\1` dans le remplacement
- les espaces juste avant une fin de ligne : `$` ne reconnaît chaque fin de ligne qu'avec le flag `re.MULTILINE`, passé à `re.sub` par son nom, `flags=`

Un squelette à copier dans la cellule de l'exercice, en remplaçant chaque `...` :

```python
propre = re.sub(r"[ \t]+", " ", texte)
propre = re.sub(r"...", r"\1", propre)
propre = re.sub(r"...", "", propre, flags=...)
print(propre)
```

### Solution

In [ ]:
propre = re.sub(r"[ \t]+", " ", texte)
propre = re.sub(r" ([,.])", r"\1", propre)
propre = re.sub(r" +$", "", propre, flags=re.MULTILINE)
print(propre)

## 5. Les chaînes entre guillemets

En Python, une chaîne s'écrit entre `"` ou entre `'`, et chacune peut contenir l'autre guillemet. Avec `re.finditer`, affichez le contenu de chaque chaîne de la ligne de code `ligne`, une par ligne. Vous devez obtenir :

```
Bonjour
à tous
l'été
il dit "oui"
```

Le motif `["'](.*?)["']` ne convient pas : il s'arrête à l'apostrophe de `l'été`. Capturez le guillemet ouvrant dans un groupe, et exigez le même pour fermer avec une référence arrière.

In [ ]:
ligne = """print("Bonjour", 'à tous', "l'été", 'il dit "oui"')"""

# Votre code ici

### Conseils supplémentaires

- dans un motif, `\1` reconnaît exactement le texte qu'a capturé le premier groupe : `(["'])` capture le guillemet ouvrant, et `\1` exige le même pour fermer
- le contenu de la chaîne est le deuxième groupe, `match.group(2)`, une répétition non gourmande `.*?` qui s'arrête au premier guillemet fermant (voir l'exercice 3)

Un squelette à copier dans la cellule de l'exercice, en remplaçant chaque `...` :

```python
for match in re.finditer(r"""...""", ligne):
  print(...)
```

### Solution

In [ ]:
for match in re.finditer(r"""(["'])(.*?)\1""", ligne):
  print(match.group(2))

## 6. Des tentatives d'intrusion

La cellule suivante écrit `auth.log`, le journal des connexions d'un serveur : chaque ligne donne la date, la machine, le service et son numéro de processus, puis un message. Une tentative de connexion SSH échouée s'écrit :

```
Oct 12 10:30:02 web01 sshd[3118]: Failed password for root from 203.0.113.7 port 52144 ssh2
Oct 12 10:31:57 web01 sshd[3120]: Failed password for invalid user admin from 203.0.113.7 port 52160 ssh2
```

`invalid user` précède un compte qui n'existe pas sur le serveur. Lisez `auth.log` dans `journal`. Avec `re.finditer` et des groupes nommés `user` et `ip`, comptez les échecs de chaque adresse dans le dictionnaire `echecs`, et rangez les comptes que vise chaque adresse dans le dictionnaire d'ensembles `comptes`. Affichez enfin, dans l'ordre de leur premier échec, les adresses qui ont au moins trois échecs, avec leur nombre d'échecs et la liste triée de leurs comptes visés. Vous devez obtenir :

```
203.0.113.7 : 7 échecs, comptes ['admin', 'oracle', 'root']
198.51.100.23 : 4 échecs, comptes ['git', 'root', 'ubuntu']
```

In [ ]:
import re
from pathlib import Path

Path("auth.log").write_text("""\
Oct 12 10:30:02 web01 sshd[3118]: Failed password for root from 203.0.113.7 port 52144 ssh2
Oct 12 10:30:05 web01 sshd[3118]: Failed password for root from 203.0.113.7 port 52144 ssh2
Oct 12 10:30:09 web01 sshd[3118]: Failed password for root from 203.0.113.7 port 52144 ssh2
Oct 12 10:30:11 web01 sshd[3118]: error: maximum authentication attempts exceeded for root from 203.0.113.7 port 52144 ssh2 [preauth]
Oct 12 10:31:57 web01 sshd[3120]: Failed password for invalid user admin from 203.0.113.7 port 52160 ssh2
Oct 12 10:32:03 web01 sshd[3120]: Failed password for invalid user admin from 203.0.113.7 port 52160 ssh2
Oct 12 10:33:40 web01 sshd[3125]: Failed password for invalid user oracle from 203.0.113.7 port 52188 ssh2
Oct 12 10:45:00 web01 CRON[3190]: pam_unix(cron:session): session opened for user root by (uid=0)
Oct 12 10:45:01 web01 CRON[3190]: pam_unix(cron:session): session closed for user root
Oct 12 11:02:13 web01 sshd[3241]: Failed password for carole from 192.0.2.15 port 40102 ssh2
Oct 12 11:02:19 web01 sshd[3241]: Accepted password for carole from 192.0.2.15 port 40102 ssh2
Oct 12 11:02:19 web01 sshd[3241]: pam_unix(sshd:session): session opened for user carole by (uid=0)
Oct 12 11:12:40 web01 sshd[3302]: Accepted publickey for bruno from 192.0.2.20 port 40112 ssh2
Oct 12 11:12:40 web01 sshd[3302]: pam_unix(sshd:session): session opened for user bruno by (uid=0)
Oct 12 11:20:31 web01 sshd[3330]: Invalid user ubuntu from 198.51.100.23 port 51000
Oct 12 11:20:33 web01 sshd[3330]: Failed password for invalid user ubuntu from 198.51.100.23 port 51000 ssh2
Oct 12 11:20:38 web01 sshd[3330]: Failed password for invalid user ubuntu from 198.51.100.23 port 51000 ssh2
Oct 12 11:21:02 web01 sshd[3333]: Failed password for root from 198.51.100.23 port 51012 ssh2
Oct 12 11:21:05 web01 sshd[3333]: Connection closed by authenticating user root 198.51.100.23 port 51012 [preauth]
Oct 12 11:48:17 web01 sshd[3360]: Failed password for invalid user test from 203.0.113.99 port 60122 ssh2
Oct 12 11:48:21 web01 sshd[3360]: Failed password for invalid user test from 203.0.113.99 port 60122 ssh2
Oct 12 11:48:22 web01 sshd[3360]: Disconnected from invalid user test 203.0.113.99 port 60122 [preauth]
Oct 12 12:03:44 web01 sshd[3412]: Failed password for root from 203.0.113.7 port 52301 ssh2
Oct 12 12:15:09 web01 sshd[3415]: Received disconnect from 192.0.2.20 port 40112:11: disconnected by user
Oct 12 12:15:09 web01 sshd[3415]: pam_unix(sshd:session): session closed for user bruno
Oct 12 12:30:00 web01 sudo[3501]:   carole : TTY=pts/0 ; PWD=/home/carole ; USER=root ; COMMAND=/usr/bin/apt update
Oct 12 13:05:51 web01 sshd[3577]: Failed password for invalid user git from 198.51.100.23 port 51340 ssh2
Oct 12 13:06:02 web01 sshd[3580]: Accepted publickey for carole from 192.0.2.15 port 40250 ssh2
""")

# Votre code ici

### Conseils supplémentaires

- `Path("auth.log").read_text()` renvoie le contenu du fichier
- `Failed password for (?:invalid user )?(?P<user>\S+) from (?P<ip>\S+)` reconnaît les deux formes d'échec : le groupe non capturant rend `invalid user ` facultatif sans ajouter de groupe. Les autres lignes, connexions acceptées ou `error: maximum authentication attempts exceeded`, ne sont pas des échecs de mot de passe
- `match["ip"]` et `match["user"]` donnent le texte des groupes nommés
- `echecs.get(ip, 0)` vaut `0` pour une adresse pas encore rencontrée ; un dictionnaire garde l'ordre d'insertion de ses clés, donc celui des premiers échecs
- `sorted(ensemble)` renvoie la liste triée des éléments d'un ensemble

Un squelette à copier dans la cellule de l'exercice, en remplaçant chaque `...` :

```python
journal = Path("auth.log").read_text()
echecs = {}
comptes = {}
for match in re.finditer(r"...", journal):
  ip = ...
  echecs[ip] = echecs.get(ip, 0) + 1
  if ip not in comptes:
    comptes[ip] = set()
  comptes[ip].add(...)

for ip, nombre in echecs.items():
  if nombre >= 3:
    print(f"{ip} : {nombre} échecs, comptes {sorted(comptes[ip])}")
```

### Solution

In [ ]:
journal = Path("auth.log").read_text()
echecs = {}
comptes = {}
for match in re.finditer(
  r"Failed password for (?:invalid user )?(?P<user>\S+) from (?P<ip>\S+)", journal
):
  ip = match["ip"]
  echecs[ip] = echecs.get(ip, 0) + 1
  if ip not in comptes:
    comptes[ip] = set()
  comptes[ip].add(match["user"])

for ip, nombre in echecs.items():
  if nombre >= 3:
    print(f"{ip} : {nombre} échecs, comptes {sorted(comptes[ip])}")

## 7. Un corpus d'e-mails frauduleux

Le corpus CLAIR rassemble 3 976 e-mails d'arnaque « à la nigériane » (fraude 419), reçus de 1998 à 2007, à la suite les uns des autres dans un même fichier. Le fichier `debug_file.txt` n'en contient que deux, pour mettre le code au point. Source : Radev (2008), *CLAIR collection of fraud email*, ACL Data and Code Repository, ADCR2008T001, publié sur https://www.kaggle.com/datasets/rtatman/fraudulent-email-corpus sous licence CC BY-SA 4.0.

Chaque e-mail commence par une ligne `From r`, puis viennent ses en-têtes, dont :

- `From` : l'émetteur (parfois vide)
- `Reply-To` : l'adresse à laquelle répondre
- `To` : le destinataire (parfois tronqué pour l'anonymat)
- `Date` : la date d'envoi
- `Subject` : l'objet du message
- `Content-Type` : le type du contenu et son encodage

et enfin, après une ligne vide, le corps du message.

In [ ]:
!wget -q https://github.com/shuuchuu/datasets/raw/refs/heads/main/ghsparse.sh -O /tmp/ghsparse.sh && bash /tmp/ghsparse.sh fraud-emails

Les fichiers sont encodés en `windows-1252` : la cellule suivante les lit dans `texte_debug` et `texte_corpus`, et affiche le début du corpus.

In [ ]:
import re
from pathlib import Path

texte_debug = Path("fraud-emails/debug_file.txt").read_text(encoding="windows-1252")
texte_corpus = Path("fraud-emails/fradulent_emails.txt").read_text(
  encoding="windows-1252"
)
print(texte_corpus[:1500])

### 7.1 Découper le corpus en e-mails

Écrivez `decoupe_emails(texte)`, qui renvoie la liste des e-mails de `texte` : découpez-le avec `re.split` à chaque ligne qui commence par `From r`. Testez-la sur `texte_debug`, qui doit donner 2 e-mails, puis rangez dans `emails` ceux de `texte_corpus` : 3 976.

**Documentation utile**

- [`re.split`](https://docs.python.org/fr/3/library/re.html#re.split)

In [ ]:
def decoupe_emails(texte: str) -> list[str]: ...


# Votre code ici

#### Conseils supplémentaires

- `^` ne reconnaît chaque début de ligne qu'avec le flag `re.MULTILINE`, à passer par son nom, `flags=` (le passer en position est obsolète depuis Python 3.13)
- le fichier commence par une ligne `From r` : le premier morceau que renvoie `re.split`, ce qui la précède, est une chaîne vide, à retirer avec `[1:]`

Un squelette à copier dans la cellule de l'exercice, en remplaçant chaque `...` :

```python
def decoupe_emails(texte: str) -> list[str]:
  return re.split(r"...", texte, flags=...)[1:]


print(len(decoupe_emails(texte_debug)))
emails = decoupe_emails(texte_corpus)
print(len(emails))
```

#### Solution

In [ ]:
def decoupe_emails(texte: str) -> list[str]:
  return re.split(r"^From r", texte, flags=re.MULTILINE)[1:]


print(len(decoupe_emails(texte_debug)))
emails = decoupe_emails(texte_corpus)
print(len(emails))

### 7.2 Les domaines des adresses de réponse

Les lignes d'en-tête `Reply-To:` donnent l'adresse à laquelle répondre, par exemple `Reply-To: james_ngola2002@maktoob.com`.

1. Avec `re.findall` et un motif qui n'a qu'**un** groupe, construisez `domaines`, la liste des domaines (ce qui suit le `@`) des lignes `Reply-To:` de `texte_corpus`.
2. Avec un motif à **deux** groupes, construisez `paires`, la liste des paires (utilisateur, domaine) de ces mêmes lignes.

Affichez le nombre d'éléments et les trois premiers de chaque liste. Vous devez obtenir :

```
2558 ['maktoob.com', 'epatra.com', 'www.com']
2556 [('james_ngola2002', 'maktoob.com'), ('obong_715', 'epatra.com'), ('m_abacha03', 'www.com')]
```

Que renvoie `re.findall` dans chaque cas ?

In [ ]:
# Votre code ici

#### Conseils supplémentaires

- un domaine est une suite de mots séparés par des points : `[\w-]+(?:\.[\w-]+)+`, où le groupe non capturant `(?:...)` répète `.mot` sans ajouter de groupe au résultat de `re.findall`
- `^Reply-To:.*?@` va du début de la ligne au premier `@` : `.*?` est non gourmand (voir l'exercice 3), et `flags=re.MULTILINE` fait reconnaître à `^` chaque début de ligne
- l'utilisateur est une suite de caractères de mot, de points et de tirets, `[\w.-]+`, juste avant le `@` ; avec `.*` gourmand à la place de `.*?`, il ne garderait que son dernier caractère
- `liste[:3]` garde les trois premiers éléments d'une liste

Un squelette à copier dans la cellule de l'exercice, en remplaçant chaque `...` :

```python
domaines = re.findall(r"...", texte_corpus, flags=re.MULTILINE)
print(len(domaines), domaines[:3])
paires = re.findall(r"...", texte_corpus, flags=re.MULTILINE)
print(len(paires), paires[:3])
```

#### Solution

In [ ]:
domaines = re.findall(
  r"^Reply-To:.*?@([\w-]+(?:\.[\w-]+)+)", texte_corpus, flags=re.MULTILINE
)
print(len(domaines), domaines[:3])
paires = re.findall(
  r"^Reply-To:.*?([\w.-]+)@([\w-]+(?:\.[\w-]+)+)", texte_corpus, flags=re.MULTILINE
)
print(len(paires), paires[:3])

# Avec un groupe, re.findall renvoie la liste des textes de ce groupe ; avec deux,
# la liste des tuples de leurs textes ; jamais le texte trouvé entier, la ligne
# depuis « Reply-To: ». Les deux paires de moins : des utilisateurs entre
# guillemets ("dr..abdull.rasaq"@caramail.com), qu'exclut [\w.-]+ juste avant @.

### 7.3 Émetteurs et adresses

Une ligne `From:` a souvent la forme `From: "Mr. Ben Suleman" <bensul2004nng@spinfinder.com>` : un nom, parfois entre guillemets, puis une adresse entre chevrons.

1. Écrivez `nettoie_nom(nom)`, qui renvoie `nom` sans ses `'` ni ses `"`, ni ses espaces au début et à la fin : `nettoie_nom(' "Mr. Ben Suleman" ')` renvoie `'Mr. Ben Suleman'`.
2. Construisez le dictionnaire `emetteurs`, qui associe à chaque nom d'émetteur, passé par `nettoie_nom`, l'ensemble de ses adresses. Gardez les lignes de la forme `nom <adresse>` dont l'adresse est plausible (des caractères sans blanc, un `@`, puis un domaine avec un point) et le nom non vide ; ignorez les autres, qui n'ont qu'une adresse ou qu'un nom.
3. Affichez le nombre de lignes `From:` gardées et ignorées, le nombre d'émetteurs, et celui des émetteurs qui ont plusieurs adresses. Avec le motif des conseils, vous devez trouver 3 190 lignes gardées, 598 ignorées, et 2 407 émetteurs, dont 162 ont plusieurs adresses.

In [ ]:
def nettoie_nom(nom: str) -> str: ...


# Votre code ici

#### Conseils supplémentaires

- dans `nettoie_nom`, `re.sub` avec la classe de caractères `['"]` supprime les guillemets, et la méthode `strip` des chaînes les espaces au début et à la fin
- un motif pour la ligne entière, avec un groupe pour le nom et un pour l'adresse : `^From:(.*)<(\w\S*@\S+\.\w+)>$`, toujours avec `re.MULTILINE`
- `re.finditer` donne chaque ligne reconnue, dont `match.group(1)` et `match.group(2)` sont le nom et l'adresse
- le nombre de lignes `From:` : la longueur de la liste que renvoie `re.findall(r"^From:", texte_corpus, flags=re.MULTILINE)`

Un squelette à copier dans la cellule de l'exercice, en remplaçant chaque `...` :

```python
def nettoie_nom(nom: str) -> str:
  return re.sub(..., "", nom).strip()


emetteurs = {}
gardees = 0
for match in re.finditer(r"...", texte_corpus, flags=re.MULTILINE):
  nom = nettoie_nom(match.group(1))
  if nom == "":
    continue
  gardees += 1
  if nom not in emetteurs:
    emetteurs[nom] = set()
  emetteurs[nom].add(...)

total = len(re.findall(r"^From:", texte_corpus, flags=re.MULTILINE))
plusieurs = [nom for nom, adresses in emetteurs.items() if len(adresses) > 1]
print(f"{gardees} lignes gardées, {total - gardees} ignorées")
print(f"{len(emetteurs)} émetteurs, dont {len(plusieurs)} avec plusieurs adresses")
```

#### Solution

In [ ]:
def nettoie_nom(nom: str) -> str:
  return re.sub(r"['\"]", "", nom).strip()


print(repr(nettoie_nom(' "Mr. Ben Suleman" ')))

emetteurs = {}
gardees = 0
for match in re.finditer(
  r"^From:(.*)<(\w\S*@\S+\.\w+)>$", texte_corpus, flags=re.MULTILINE
):
  nom = nettoie_nom(match.group(1))
  if nom == "":
    continue
  gardees += 1
  if nom not in emetteurs:
    emetteurs[nom] = set()
  emetteurs[nom].add(match.group(2))

total = len(re.findall(r"^From:", texte_corpus, flags=re.MULTILINE))
plusieurs = [nom for nom, adresses in emetteurs.items() if len(adresses) > 1]
print(f"{gardees} lignes gardées, {total - gardees} ignorées")
print(f"{len(emetteurs)} émetteurs, dont {len(plusieurs)} avec plusieurs adresses")

### 7.4 Les e-mails par année

Pour chaque e-mail de `emails`, cherchez l'année de son premier en-tête `Date:` avec `re.search` et un groupe nommé `annee`, et comptez les e-mails de chaque année dans le dictionnaire `par_annee`. Affichez les années dans l'ordre, chacune avec son nombre d'e-mails, puis le nombre d'e-mails sans année. Vous devez trouver 1 170 e-mails en 2006, l'année la plus chargée, et 566 sans année.

Le corpus va de 1998 à 2007 : d'où viennent les autres années ?

In [ ]:
# Votre code ici

#### Conseils supplémentaires

- un groupe nommé s'écrit `(?P<annee>...)`, et `match.group("annee")` donne ce qu'il a capturé
- dans `^Date:.*?(?P<annee>\d{4})`, la répétition non gourmande s'arrête au premier nombre de quatre chiffres de la ligne : `2002` dans `Date: Thu, 31 Oct 2002 05:10:00`
- `re.search` renvoie le premier résultat, ou `None` quand l'e-mail n'a pas d'en-tête `Date:` avec une année
- `par_annee.get(annee, 0)` vaut `0` pour une année pas encore rencontrée, et `sorted(par_annee.items())` donne les paires `(année, nombre)` dans l'ordre des années

Un squelette à copier dans la cellule de l'exercice, en remplaçant chaque `...` :

```python
par_annee = {}
sans_annee = 0
for email in emails:
  match = re.search(r"...", email, flags=re.MULTILINE)
  if match is None:
    sans_annee += 1
  else:
    annee = ...
    par_annee[annee] = par_annee.get(annee, 0) + 1

for annee, nombre in sorted(par_annee.items()):
  print(annee, nombre)
print("Sans année :", sans_annee)
```

#### Solution

In [ ]:
par_annee = {}
sans_annee = 0
for email in emails:
  match = re.search(r"^Date:.*?(?P<annee>\d{4})", email, flags=re.MULTILINE)
  if match is None:
    sans_annee += 1
  else:
    annee = match.group("annee")
    par_annee[annee] = par_annee.get(annee, 0) + 1

for annee, nombre in sorted(par_annee.items()):
  print(annee, nombre)
print("Sans année :", sans_annee)

# Les 15 e-mails datés de 1980, 1990, 2015 ou 2082 viennent d'ordinateurs mal mis à
# l'heure (ou d'en-têtes falsifiés) : une analyse par année les écarte.

## 8. Nettoyer un texte accentué (facultatif)

Avant d'analyser un texte (compter ses mots, le comparer à d'autres), on le *normalise* : minuscules, sans accents ni ponctuation. Ce qu'est un caractère de mot dépend alors des flags.

**1.** Affichez les mots de `texte`, `re.findall(r"\w+", texte)`, puis les mêmes avec `flags=re.ASCII`. Quelle version découpe mal les mots, et pourquoi ?

**2.** Écrivez `sans_accents(texte)`, qui renvoie `texte` sans ses accents. `unicodedata.normalize("NFD", texte)` décompose chaque lettre accentuée en sa lettre de base suivie d'un accent *combinant* (`"é"` devient `"e"` suivi de `"\u0301"`), et les accents combinants occupent l'intervalle de `\u0300` à `\u036f` : supprimez-les avec `re.sub`. `print(sans_accents(texte))` doit afficher :

```
Elodie a vu « Œdipe roi » au theatre : 3 cafes, 2 cremes brulees… Ca coute 12,50 € !
```

**3.** Écrivez `nettoie(texte)`, qui renvoie `texte` en minuscules, sans accents, chaque suite de caractères qui ne sont pas des caractères de mot remplacée par une seule espace, et sans espace au début ni à la fin. `print(nettoie(texte))` doit afficher :

```
elodie a vu œdipe roi au theatre 3 cafes 2 cremes brulees ca coute 12 50
```

Le `œ` est resté : pourquoi ? Que deviendrait `œdipe` avec `re.ASCII` à l'étape 3 ?

In [ ]:
import re
import unicodedata

texte = (
  "Élodie a vu « Œdipe roi » au théâtre : 3 cafés, 2 crèmes brûlées… Ça coûte 12,50 € !"
)

# Votre code ici

### Conseils supplémentaires

- avec `re.ASCII`, `\w` ne reconnaît que `[a-zA-Z0-9_]` : chaque lettre accentuée coupe le mot
- dans un motif, `[\u0300-\u036f]` est la classe de caractères de cet intervalle : `re` comprend lui-même la notation `\uXXXX`, même dans une chaîne brute
- `nettoie` enchaîne `sans_accents`, la méthode `lower` des chaînes, `re.sub` avec `\W+`, et la méthode `strip`

Un squelette à copier dans la cellule de l'exercice, en remplaçant chaque `...` :

```python
print(re.findall(r"\w+", texte))
print(re.findall(r"\w+", texte, flags=...))


def sans_accents(texte: str) -> str:
  return re.sub(r"...", "", unicodedata.normalize("NFD", texte))


def nettoie(texte: str) -> str:
  return re.sub(r"...", " ", ...).strip()


print(sans_accents(texte))
print(nettoie(texte))
```

### Solution

In [ ]:
print(re.findall(r"\w+", texte))
print(re.findall(r"\w+", texte, flags=re.ASCII))
# Avec re.ASCII, une lettre accentuée n'est plus un caractère de mot : « Élodie »
# devient « lodie », et « théâtre » se coupe en « th » et « tre ».


def sans_accents(texte: str) -> str:
  return re.sub(r"[\u0300-\u036f]", "", unicodedata.normalize("NFD", texte))


def nettoie(texte: str) -> str:
  return re.sub(r"\W+", " ", sans_accents(texte).lower()).strip()


print(sans_accents(texte))
print(nettoie(texte))
# « œ » est une ligature, pas une lettre accentuée : NFD ne la décompose pas, il
# faut la remplacer à la main (texte.replace("œ", "oe")). Avec re.ASCII, \W la
# prendrait pour un séparateur : « œdipe » deviendrait « dipe ».